<h1 style="color: #008080;">Código para generar base de datos "stagin" desde el Observatorio de Compra Pública de Costa Rica.</h1>

Descarga los archivos ZIP mensuales del repositorio público del Observatorio de Compra Pública de Costa Rica (desde Azure Blob Storage)
y extrae los datos disponibles de SICOP, para luego ser transformados en distintos dataframes a cargar en la base de datos.

URL sitio web:
    https://www.observatoriocomprapublica.go.cr/descargas-sicop/
    
URL base de descarga:
    https://dlsaobservatorioprod.blob.core.windows.net/fs-synapse-observatorio-produccion/Zip/yyyymm.zip

Instalación de paquetes en terminal:
`pip install requests`
`pip install python-dateutil`
`pip install psycopg2`

Importación de las librerías necesarias:

In [1]:
import uuid
import requests
import io
import csv
import zipfile
import psycopg2
import logging
import time
from datetime import date, datetime, timedelta
from dateutil.relativedelta import relativedelta
from pathlib import Path
from typing import Optional
from io import BytesIO
from io import StringIO
from io import TextIOWrapper

Creamos una sección inicial donde se configuran todos los parámetros importantes del código ETL.

Se define la cantidad de meses de datos por consultar en la dirección de la página web para crear los datasets iniciales.

Definimos un diccionario con los archivos .csv y columnas de cada archivo que son de interés para crear la base de datos y realizar análisis sobre licitaciones públicas.

In [2]:
# Configuración general del proyecto SICOP ETL

BASE_URL = "https://dlsaobservatorioprod.blob.core.windows.net/fs-synapse-observatorio-produccion/Zip"
MESES_A_DESCARGAR = 7 #Número de meses a descargar, incluyendo el mes actual.
#12 meses puede tardar unos 7 minutos en descargar con una velocidad de 10Mbps. 24 meses puede tardar unos 15 minutos.

#LECTURA DE CSV
CSV_ENCODING = "utf-8-sig"
CSV_SEPARATOR = ";"
CHUNK_SIZE = 100000
LOW_MEMORY = False

#DATOS DE LA BASE DE DATOS EN POSTGRESQL
DATABASE = {
    "host": "localhost",
    "port": 5432,
    "database": "proyecto_sicop_v1",
    "user": "postgres",
    "password": "YourPostgreSQLPassword" ## colocar acá la contraseña de su base de datos PostgreSQL.
}


#CONFIGURACIÓN DE CADA ARCHIVO .CSV
CSV_CONFIG = {

    "InstitucionesRegistradas.csv": {
        "tabla": "staging.stg_instituciones",
        "columnas": [
            "CEDULA",
            "NOMBRE_INSTITUCION",
            "ZONA_GEO_INST",
            "FECHA_INGRESO",
        ]
    },

    "Proveedores.csv": {
        "tabla": "staging.stg_proveedores",
        "columnas": [
            "CEDULA_PROVEEDOR",
            "NOMBRE_PROVEEDOR",
            "TIPO_PROVEEDOR",
            "TAMAÑO_PROVEEDOR", ## TAMAÑO
            "FECHA_CONSTITUCION",
            "zona_geo_prov",
            "fecha_registro"
        ]
    },

    "DetalleCarteles.csv": {
        "tabla": "staging.stg_carteles",
        "columnas": [
            "NRO_SICOP",
            "CEDULA_INSTITUCION",
            "FECHA_PUBLICACION",
            "NRO_PROCEDIMIENTO",
            "TIPO_PROCEDIMIENTO",
            "MODALIDAD_PROCEDIMIENTO",
            "CARTEL_STAT", ## STATUS_CARTEL
            "CARTEL_NM", ## NOMBRE_CARTEL
            "FECHAH_APERTURA",
            "CLAS_OBJ", ## CLASIFICACION_CARTEL
            "MONTO_EST" ## Puede contener NULL, ya que no todos los carteles tienen un monto estimado.
        ]
    },

    "DetalleLineaCartel.csv": {
        "tabla": "staging.stg_lineas_carteles",
        "columnas": [
            "NRO_SICOP",
            "NUMERO_LINEA",
            "NUMERO_PARTIDA",
            "CANTIDAD_SOLICITADA",
            "PRECIO_UNITARIO_ESTIMADO",
            "TIPO_MONEDA",
            "TIPO_CAMBIO_CRC",
            "CODIGO_IDENTIFICACION",
            "MONTO_RESERVADO", ## Se refiere al total de cada línea (Cantidad X Precio unitario) en la moneda especificada.
            "DESC_LINEA"
        ]
    },

    "Ofertas.csv": {
        "tabla": "staging.stg_ofertas",
        "columnas": [
            "NRO_SICOP", ## Opcional, puede eliminarse.
            "NRO_OFERTA",
            "CEDULA_PROVEEDOR",
            "FECHA_PRESENTA_OFERTA",
            "TIPO_OFERTA" 
        ]
    },

    "LineasOfertadas.csv": {
        "tabla": "staging.stg_lineas_ofertadas",
        "columnas": [
            "NRO_SICOP", ## Opcional, puede eliminarse.
            "NRO_OFERTA",
            "NRO_LINEA",
            "CODIGO_PRODUCTO_CL",
            "CANTIDAD_OFERTADA",
            "PRECIO_UNITARIO_OFERTADO",
            "TIPO_MONEDA",
            "TIPO_CAMBIO_CRC" 
        ]
    },

    "ProcedimientoAdjudicacion.csv": {
        "tabla": "staging.stg_procedimientos_adjudicacion",
        "columnas": [
            "NRO_SICOP",
            "CEDULA", ## CEDULA INSTITUCIÓN
            "NUMERO_PROCEDIMIENTO",
            "DESCR_PROCEDIMIENTO",
            "LINEA",
            "PROD_ID", ##CODIGO DE PRODUCTO
            "FECHA_ADJUD_FIRME",
            "MONTO_ADJU_LINEA" ##MONTO_ADJUDICADO_LINEA
        ]
    },

    "LineasAdjudicadas.csv": {
        "tabla": "staging.stg_lineas_adjudicadas",
        "columnas": [
            "NRO_SICOP",
            "NRO_OFERTA",
            "NRO_LINEA",
            "CODIGO_PRODUCTO",
            "CEDULA_PROVEEDOR",
            "CANTIDAD_ADJUDICADA",
            "PRECIO_UNITARIO_ADJUDICADO",
            "IVA",
            "OTROS_IMPUESTOS",
            "ACARREOS",
            "TIPO_MONEDA",
            "TIPO_CAMBIO_CRC" 
        ]
    }
}

Función para crear una lista con los meses por consultar de información.
Cada respuesta de la página web entregará los datos correspondientes un mes específico de licitaciones.

In [3]:
def meses_descarga(n: int) -> list[str]:
    #Genera lista de códigos yyyymm para los últimos n meses.
    hoy = date.today()
    meses = []
    for i in range(n):
        d = hoy - relativedelta(months= (i)) # Si no desea incluir el mes actual, use i+1 en lugar de i.
        meses.append(d.strftime("%Y%m"))
    return meses

#meses = meses_descarga(MESES_A_DESCARGAR) # comentario para prueba de descarga de 2 meses
#print("Meses a descargar:", meses)

Se definieron tres (3) funciones para administrar la conexión con Postgre SQL. Esta serán las encargadas de: Crear, probar y cerrar la conexión con la base de datos.

In [4]:
#Administración de la conexión PostgreSQL.
from __future__ import annotations
logger = logging.getLogger(__name__)

def get_connection(): #Devuelve una conexión PostgreSQL.
    conn = psycopg2.connect(
        host=DATABASE["host"],
        port=DATABASE["port"],
        database=DATABASE["database"],
        user=DATABASE["user"],
        password=DATABASE["password"]
    )

    conn.autocommit = False
    logger.info(
        "Conectado a PostgreSQL."
    )
    return conn

def test_connection(conn): #Ejecuta una consulta sencilla para verificar la conexión.
    with conn.cursor() as cursor:
        cursor.execute(
            "SELECT version();"
        )
        version = cursor.fetchone()[0]
        logger.info(version)

def close_connection(conn): #Cierra la conexión.
    if conn is not None:
        conn.close()
        logger.info(
            "Conexión cerrada."
        )


Función `descargar_zip` para solicitar archivos zip a la página web del observatorio.

In [5]:
#Descarga de archivos ZIP desde el repositorio SICOP.
from __future__ import annotations

logger = logging.getLogger(__name__)

class DownloadError(Exception):
    """Error durante la descarga de un ZIP."""

#Descarga un ZIP y devuelve su contenido en memoria.
def descargar_zip( 
    period: str,
    timeout: int = 120,
    retries: int = 3
) -> bytes:
    
    url = f"{BASE_URL}/{period}.zip"
    for attempt in range(1, retries + 1):
        try:
            logger.info("Descargando %s", url)
            response = requests.get(
                url,
                timeout=timeout
            )

            response.raise_for_status()
            logger.info(
                "%s descargado (%d bytes)",
                period,
                len(response.content)
            )
            return response.content

        except Exception as ex:
            logger.warning(
                "Intento %d/%d fallido (%s)",
                attempt,
                retries,
                ex
            )
            if attempt == retries:
                raise DownloadError(
                    f"No fue posible descargar {period}"
                ) from ex

            time.sleep(5)

Ahora definimos una función `iter_csv_files` para leer los archivos CSV contenidos en el ZIP y generar los dataframes.

La extracción se realizará a un máximo de 100,000 filas para optimizar la memoria (definido en parámetro CHUNK_SIZE).

In [6]:
#Lectura de archivos CSV contenidos en un ZIP.
"""
Extrae archivos CSV contenidos dentro de un archivo ZIP
y devuelve las filas en streaming para ser cargadas
mediante COPY hacia PostgreSQL.
"""

from __future__ import annotations
logger = logging.getLogger(__name__)


def iter_csv_files(zip_bytes: bytes, period, loaded_at, batch_id):
    """
    Recorre todos los CSV contenidos en el ZIP.
    Yields
    ------
    tuple
        filename : str
        tabla : str
        columnas : list[str]
        filas : iterator[list]
    """

    with zipfile.ZipFile(BytesIO(zip_bytes)) as archive:
        for path in archive.namelist():
            filename = path.split("/")[-1]
            if filename not in CSV_CONFIG:
                continue

            cfg = CSV_CONFIG[filename]

            logger.info(
                "Procesando %s",
                filename
            )

            with archive.open(path) as binary_file:

                text_file = TextIOWrapper(
                    binary_file,
                    encoding=CSV_ENCODING,
                    newline=""
                )

                reader = csv.DictReader(
                    text_file,
                    delimiter=CSV_SEPARATOR
                )
                #logger.info(reader.fieldnames)

                columns = cfg["columnas"]
                ##print(f"Columnas a cargar en la tabla {cfg['tabla']}: {columns}")

                missing = set(columns) - set(reader.fieldnames)
                if missing:
                    raise ValueError(
                        f"{filename}: las siguientes columnas no existen en el CSV: {sorted(missing)}"
                    )

                def row_generator():
                    for row in reader:
                        yield [
                            row.get(column, "")
                            for column in columns
                        ] + [
                        period,
                        filename,
                        loaded_at.isoformat(),
                        batch_id
                        ]

                yield (
                    filename,
                    cfg["tabla"],
                    columns,
                    row_generator()
                )

Cómo última función definimos `copy_stream` para cargar los datos por bloques a la base de datos en PostgreSQL.

In [7]:
#Carga datos hacia PostgreSQL utilizando COPY en bloques (streaming) para cada tabla.

from __future__ import annotations
logger = logging.getLogger(__name__)

def copy_stream( conn, table_name: str, columns: list[str], rows ):
    #Carga un flujo de filas utilizando COPY.
    buffer = StringIO()

    writer = csv.writer(
        buffer,
        delimiter=CSV_SEPARATOR,
        lineterminator="\n",
        quoting=csv.QUOTE_MINIMAL
    )

    current_rows = 0
    total_rows = 0
    column_list = ", ".join(columns+[
        "etl_period",
        "source_file",
        "loaded_at",
        "batch_id"
    ])

    sql = f"""
        COPY {table_name}
        ({column_list})

        FROM STDIN
        WITH (
            FORMAT CSV,
            DELIMITER '{CSV_SEPARATOR}',
            NULL ''
        )
    """

    with conn.cursor() as cursor:
        for row in rows:
            writer.writerow(row)
            current_rows += 1
            total_rows += 1

            if current_rows >= CHUNK_SIZE: #si la cantidad de filas en el buffer alcanza el tamaño del chunk, se hace un COPY y se vacía el buffer.
                buffer.seek(0)

                cursor.copy_expert(
                    sql=sql,
                    file=buffer
                )

                buffer.close()

                buffer = StringIO()

                writer = csv.writer(
                    buffer,
                    delimiter=CSV_SEPARATOR,
                    lineterminator="\n",
                    quoting=csv.QUOTE_MINIMAL
                )
                current_rows = 0

        if current_rows > 0:

            buffer.seek(0)

            cursor.copy_expert(
                sql=sql,
                file=buffer
            )

        buffer.close()

    logger.info(
        "%d filas cargadas en %s",
        total_rows,
        table_name
    )

Finalmente, ejecutamos todo el proceso de ETL, llamando cada función definida anteriormente.

In [9]:
#Proceso principal del ETL.
from __future__ import annotations

logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | "
        "%(levelname)s | "
        "%(message)s"
    )
)
logger = logging.getLogger(__name__)

meses = meses_descarga(MESES_A_DESCARGAR)

batch_id = str(uuid.uuid4()) #variable para identificar el proceso de carga en la base de datos de tablas staging mediante un id único.
loaded_at = datetime.now() #variable para registrar la fecha y hora en que se cargaron los datos.

conn = get_connection()
test_connection(conn)

try:
    for period in meses:
        logger.info("=" * 70)
        logger.info("Procesando período %s",period)

        zip_bytes = descargar_zip(period)

        for filename, tabla, columnas, filas in iter_csv_files(zip_bytes, period, loaded_at, batch_id):

            copy_stream(
                conn,
                tabla,
                columnas,
                filas
            )

        conn.commit()

        logger.info(
            "Período %s finalizado.",
            period
        )

except Exception as ex:
    logger.exception(ex)
    conn.rollback()

finally:
    close_connection(conn)


2026-07-15 22:20:01,267 | INFO | Conectado a PostgreSQL.
2026-07-15 22:20:01,271 | INFO | PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit
2026-07-15 22:20:01,273 | INFO | ======================================================================
2026-07-15 22:20:01,273 | INFO | Procesando período 202607
2026-07-15 22:20:01,274 | INFO | Descargando https://dlsaobservatorioprod.blob.core.windows.net/fs-synapse-observatorio-produccion/Zip/202607.zip
2026-07-15 22:20:07,548 | INFO | 202607 descargado (21176650 bytes)
2026-07-15 22:20:07,559 | INFO | Procesando LineasAdjudicadas.csv
2026-07-15 22:20:07,628 | INFO | 3098 filas cargadas en staging.stg_lineas_adjudicadas
2026-07-15 22:20:07,629 | INFO | Procesando LineasOfertadas.csv
2026-07-15 22:20:07,745 | INFO | 7536 filas cargadas en staging.stg_lineas_ofertadas
2026-07-15 22:20:07,746 | INFO | Procesando Ofertas.csv
2026-07-15 22:20:07,881 | INFO | 10165 filas cargadas en staging.stg_ofertas
2026-07-15 22:20:07,882 | 